# Delta Lake over Spark Connect
Run the cells in order. This notebook uses the `spark-connect` service (`sc://spark-connect:15002`) instead of connecting directly to the Spark master. The Spark Connect server is pre-configured with Delta, the Hive Metastore, and the MinIO S3A connector, so the remote client only needs `SparkSession.builder.remote(...)`.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote('sc://localhost:15002').getOrCreate()

## Create a catalog database
The Hive Metastore is the catalog service shared with the standalone Spark cluster. Catalog configuration (Delta extensions, S3A, the metastore URI) is set on the Spark Connect server, not on this client.

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS notebook_connect LOCATION 's3a://warehouse/notebook_connect'")
spark.sql('USE notebook_connect')
spark.sql('SHOW DATABASES').show()

+----------------+
|       namespace|
+----------------+
|catalog_examples|
|         crawler|
|         default|
|notebook_connect|
+----------------+



## Write, append, and query a registered table

In [ ]:
table_name = 'notebook_connect.people'
df = spark.createDataFrame([(1, 'Ada', 'engineering'), (2, 'Grace', 'data')], ['id', 'name', 'team'])
df.write.format('delta').mode('overwrite').saveAsTable(table_name)
df2 = spark.createDataFrame([(3, 'Lin', 'platform')], ['id', 'name', 'team'])
df2.write.format('delta').mode('append').saveAsTable(table_name)
spark.table(table_name).orderBy('id').show()

In [ ]:
spark.sql('SHOW TABLES IN notebook_connect').show()
spark.sql('DESCRIBE EXTENDED notebook_connect.people').filter("col_name = 'Location'").show(truncate=False)

## Schema evolution

In [ ]:
schema_table = 'notebook_connect.people_schema'
spark.createDataFrame([(1, 'Ada')], ['id', 'name']).write.format('delta').mode('overwrite').saveAsTable(schema_table)
spark.createDataFrame([(2, 'Grace', 'data')], ['id', 'name', 'team']).write.format('delta').option('mergeSchema', 'true').mode('append').saveAsTable(schema_table)
spark.table(schema_table).printSchema()

## Transaction history and time travel

In [ ]:
spark.sql('DESCRIBE HISTORY notebook_connect.people').select('version', 'operation').show()
spark.read.format('delta').option('versionAsOf', 0).table(table_name).show()

## Cleanup
Drop the example tables and database, then stop the Spark Connect session.

In [ ]:
spark.sql('DROP TABLE IF EXISTS notebook_connect.people')
spark.sql('DROP TABLE IF EXISTS notebook_connect.people_schema')
spark.sql('DROP DATABASE IF EXISTS notebook_connect')
spark.stop()